# 🤖 Meeting Ghost — Sprint Runner
Run each section in order. Each section = one sprint.

**Setup once:**
1. Runtime → Change runtime type → T4 GPU
2. Add secrets: `HF_TOKEN` and `ANTHROPIC_API_KEY`
3. Upload your audio file to `/content/`

In [1]:
# ── SETUP: Clone repo and install deps ──────────────────────────
# Run this cell ONCE per Colab session

import subprocess, sys

# Clone your repo (replace with your GitHub URL after Sprint 6)
!git clone https://github.com/LuminescentLuxe/meeting-ghost.git
%cd meeting-ghost

# Install all dependencies
!pip install -q faster-whisper torch transformers pyannote.audio \
    anthropic scikit-learn datasets accelerate python-dotenv \
    pandas tqdm gTTS pydub

print('✅ Dependencies installed')

Cloning into 'meeting-ghost'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 20 (delta 0), reused 20 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 15.15 KiB | 15.15 MiB/s, done.
/content/meeting-ghost
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.5 MB/s eta 0:00:

In [ ]:
# ── SETUP: Load secrets ─────────────────────────────────────────
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('✅ Secrets loaded')

## Sprint 1 — Transcription

In [ ]:
# Option A: Use your own audio file
# AUDIO_FILE = '/content/my_meeting.wav'

# Option B: Generate a synthetic test file (no mic needed)
from meeting_ghost.transcribe import generate_test_audio, transcribe, save_transcript

audio = generate_test_audio('/content/test_meeting.wav')
segments = transcribe(audio, model_size='base')
save_transcript(segments, '/content/transcript.json')

print('\n📄 Transcript preview:')
for s in segments:
    print(f"  [{s['start']:.1f}s] {s['text']}")

## Sprint 2 — Speaker Diarization (run after Sprint 1 works)

In [ ]:
from meeting_ghost.diarize import diarize
from meeting_ghost.merge import merge_transcript_speakers
from meeting_ghost.transcribe import load_transcript
import json

segments = load_transcript('/content/transcript.json')
speaker_segments = diarize('/content/test_meeting.wav')
labeled = merge_transcript_speakers(segments, speaker_segments)

with open('/content/labeled_transcript.json', 'w') as f:
    json.dump(labeled, f, indent=2)

print('\n👥 Labeled transcript:')
for s in labeled:
    print(f"  [{s['speaker']}] {s['text']}")

## Sprint 3 — Classification

In [ ]:
from meeting_ghost.classify import classify_transcript
import json

with open('/content/labeled_transcript.json') as f:
    labeled = json.load(f)

classified = classify_transcript(labeled)

with open('/content/classified_transcript.json', 'w') as f:
    json.dump(classified, f, indent=2)

print('\n🏷️ Classification results:')
for s in classified:
    print(f"  [{s['label']:20s}] {s['text']}")

## Sprint 4 — 2-minute Catch-up

In [ ]:
from meeting_ghost.catchup import generate_catchup
import json

with open('/content/classified_transcript.json') as f:
    classified = json.load(f)

# Replace 'Ali' with your name to test the relevance filter
catchup = generate_catchup(classified, your_name='Ali')
print(catchup)